# Train lại RF-DETR Medium với ảnh camera (dataset v5) – PBL4 tuần 2/11

Thứ tự: chạy lần lượt từ trên xuống. Runtime → Change runtime type → **T4 GPU**.

| Bước | Khi nào | Thời gian |
|---|---|---|
| 1–3. Drive, cài đặt, tải dataset | T3 | 5 phút |
| 4. Kiểm tra chia tập | T3 | 1 phút |
| 5. Train thử 1 epoch (`EPOCHS = 1`) | T3 | 10–20 phút |
| 5. Train thật (`EPOCHS = 40`, có early stopping) | T4 | vài giờ, tuỳ số ảnh |
| 6. Chấm cũ/mới trên tập test camera | T5 | 10 phút |
| 7. Đo FPS | T5 | 2 phút |

Colab ngắt giữa chừng: chạy lại bước 1–3 rồi bước 5, notebook tự train tiếp từ `last.ckpt` trên Drive.

Tham số giữ giống lần train seed 1 ở giai đoạn 1 (`rfdetr_m_seed1/training_config.json`: lr 1e-4, batch 4 × tích luỹ 4, early stopping 10 epoch); chỉ đổi dataset, `resolution` và trọng số khởi đầu.

In [ ]:
# 1. Drive và đường dẫn chung. Thư mục được chia sẻ thì bấm "Thêm lối tắt vào Drive" trước để nó nằm trong MyDrive.
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/PBL4_benchmark"                  # sửa nếu thư mục nằm chỗ khác
# Không thêm lối tắt được thì dùng đường dẫn lần train seed 1 đã dùng:
# DRIVE = "/content/drive/.shortcut-targets-by-id/1RXAfaw3ukasNvmLgNq2BxckeUl1_YBq7/PBL4_benchmark"
CKPT_SEED1 = f"{DRIVE}/rfdetr_m_seed1/checkpoint_best_total.pth"  # checkpoint đang dùng (mô hình cũ)
OUT = f"{DRIVE}/rfdetr_m_v5"                                       # checkpoint mới lưu thẳng lên Drive
RESOLUTION = 704                                                   # bội của 64 cho chắc (train đa tỉ lệ); seed 1 dùng 576
EPOCHS = 1                                                         # T3 chạy thử 1 epoch; T4 đổi thành 40

import os
print("Checkpoint cũ:", os.path.exists(CKPT_SEED1), CKPT_SEED1)

In [ ]:
# 2. Cài đặt. Nên cài đúng bản rfdetr đang chạy trên laptop nhóm trưởng (trên laptop: pip show rfdetr),
#    ví dụ: !pip install -q "rfdetr==1.11.0" roboflow
!pip install -q rfdetr roboflow
!git clone -q https://github.com/donghuyhoang/exam-behavior-monitor /content/repo 2>/dev/null || git -C /content/repo pull -q
REPO = "/content/repo/exam-behavior-monitor-setup"

import torch, rfdetr
print("rfdetr", getattr(rfdetr, "__version__", "?"), "| torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")

In [ ]:
# 3. Tải dataset v5 (bản xuất COCO, KHÔNG có bước Resize 512x512).
#    Cách A: Roboflow API. Key ở Roboflow -> Settings -> API Keys. Không dán key vào file đẩy lên GitHub.
from roboflow import Roboflow
rf = Roboflow(api_key="DAN_API_KEY_VAO_DAY")
ds = rf.workspace("TEN_WORKSPACE").project("TEN_PROJECT").version(5).download("coco", location="/content/dataset_v5")
DATASET = ds.location

#    Cách B: A đã để file zip trên Drive -> bỏ dấu # ở hai dòng dưới, thêm # vào ba dòng Cách A.
# !unzip -q -o "{DRIVE}/dataset_v5.zip" -d /content/dataset_v5
# DATASET = "/content/dataset_v5"
print(os.listdir(DATASET))

In [ ]:
# 4. Kiểm tra chia tập theo video: test không được chung video với train/valid; ảnh không bị Resize 512x512.
!python {REPO}/tools/check_split.py "{DATASET}"

In [ ]:
# 5. Train. Tiếp tục từ checkpoint seed 1 (đã quen 3 lớp student/phone/calculator).
#    Nếu báo lỗi khi nạp checkpoint: bỏ pretrain_weights=... để train từ trọng số COCO như giai đoạn 1.
#    Hết bộ nhớ GPU: batch_size=2, grad_accum_steps=8 (giữ tích = 16).
from rfdetr import RFDETRMedium

os.makedirs(OUT, exist_ok=True)
last = f"{OUT}/last.ckpt"                 # bản rfdetr của nhóm lưu điểm train tiếp ở đây (giống seed 1)
resume = last if os.path.exists(last) else None
print("Train tiếp từ", resume) if resume else print("Train mới")

model = RFDETRMedium(pretrain_weights=CKPT_SEED1, resolution=RESOLUTION)
model.train(
    dataset_dir=DATASET,
    epochs=EPOCHS,
    batch_size=4,
    grad_accum_steps=4,
    lr=1e-4,
    output_dir=OUT,
    early_stopping=True,
    early_stopping_patience=10,
    seed=1,
    resume=resume,
)
print(sorted(os.listdir(OUT)))

In [ ]:
# 6. Chấm mô hình cũ và mới trên CÙNG tập test camera (buổi 1 + nghiệm thu 1), ra một bảng so sánh.
import json
TEST = f"{DRIVE}/test_camera_v2"            # thư mục ảnh + _annotations.coco.json (A chuẩn bị)
PRED = f"{DRIVE}/predictions_camera_v2"
os.makedirs(PRED, exist_ok=True)

!python {REPO}/tools/export_preds_rfdetr.py --images "{TEST}" --weights "{CKPT_SEED1}" --out "{PRED}/pred_rfdetr_m_cu.json"
!python {REPO}/tools/export_preds_rfdetr.py --images "{TEST}" --weights "{OUT}/checkpoint_best_total.pth" --resolution {RESOLUTION} --out "{PRED}/pred_rfdetr_m_v5.json"

json.dump({"model": "RF-DETR-M cũ (seed 1)", "img_size": 576}, open(f"{PRED}/info_rfdetr_m_cu.json", "w"), ensure_ascii=False)
json.dump({"model": "RF-DETR-M v5 (thêm ảnh camera)", "img_size": RESOLUTION}, open(f"{PRED}/info_rfdetr_m_v5.json", "w"), ensure_ascii=False)
!pip install -q pycocotools
!python "{DRIVE}/evaluate_detections.py" --gt "{TEST}/_annotations.coco.json" --pred "{PRED}/pred_*.json" --out "{PRED}"

In [ ]:
# 7. Đo FPS của checkpoint mới trên T4 (nhóm trưởng đo lại trên laptop bằng cùng lệnh).
!mkdir -p {REPO}/weights && cp "{OUT}/checkpoint_best_total.pth" {REPO}/weights/
!sed -i "s/^  resolution: .*/  resolution: {RESOLUTION}/" {REPO}/config.yaml
!cd {REPO} && python -m ai_service.bench_fps --video "{DRIVE}/videos/buoi_quay_2/buoi_quay2.mp4"